# ViTVS Audio Denoising - Colab Runner
Notebook ini digunakan untuk menjalankan training, validation, dan testing secara langsung di Google Colab dengan melakukan *clone* terhadap repositori utama.


### 1. Persiapan Lingkungan (Mount Drive & Clone Repo)


In [ ]:
# Mount Google Drive (Opsional jika dataset/checkpoint ada di GDrive)
from google.colab import drive
drive.mount('/content/drive')

# Contoh jika repo sudah di-push ke GitHub, ganti URL di bawah:
# !git clone https://github.com/username/bird-denoising-new.git
# %cd bird-denoising-new

# Install dependencies dari requirements.txt
!pip install -q -r requirements.txt


### 2. Persiapan Dataset
Sesuai dataset Zenodo, pastikan Anda membagi folder dataset menjadi Train, Valid, dan Test.


In [ ]:
import os
import shutil
from tqdm.auto import tqdm

# Membuat struktur direktori lokal untuk menampung data
os.makedirs('data/train/noisy', exist_ok=True)
os.makedirs('data/train/clean', exist_ok=True)
os.makedirs('data/valid/noisy', exist_ok=True)
os.makedirs('data/valid/clean', exist_ok=True)
os.makedirs('data/test/noisy', exist_ok=True)
os.makedirs('data/test/clean', exist_ok=True)
os.makedirs('checkpoints', exist_ok=True)

def copy_600_files(src_noisy, src_clean, dst_noisy, dst_clean):
    try:
        # Ambil 600 file berformat .wav yang ada di folder clean
        files = sorted([f for f in os.listdir(src_clean) if f.endswith('.wav')])[:600]
        print(f"Memulai proses copy ke {dst_clean} dan {dst_noisy}...")
        for f in tqdm(files, desc=f"Menyalin ke {os.path.basename(dst_clean)}"):
            shutil.copy(os.path.join(src_noisy, f), dst_noisy)
            shutil.copy(os.path.join(src_clean, f), dst_clean)
        print(f"Berhasil menyalin {len(files)} file!\n")
    except FileNotFoundError as e:
        print(f"Direktori tidak ditemukan: {e}\n")

# Opsional: Mengambil 600 data saja dengan nama yang sama untuk train dan valid
# Menyalin untuk data train
copy_600_files(
    '/content/drive/MyDrive/dataset/unzip/Raw_audios',      # Raw -> train/noisy
    '/content/drive/MyDrive/dataset/unzip/Denoised_audios', # Denoised -> train/clean
    'data/train/noisy',
    'data/train/clean'
)

# Menyalin untuk data valid
copy_600_files(
    '/content/drive/MyDrive/dataset/unzip/valid/Raw_audios',        # Raw -> valid/noisy
    '/content/drive/MyDrive/dataset/unzip/valid/Denoised_audios',   # Denoised -> valid/clean
    'data/valid/noisy',
    'data/valid/clean'
)


### 3. Training & Validation Model
Jalankan `train.py`. Model sekarang akan memantau `val_loss` untuk menyimpan checkpoint terbaik. Fitur *resume training* juga masih aktif jika terputus.


In [ ]:
# Jalankan proses training beserta direktori validasi
# Anda dapat menggunakan default path, atau spesifikasikan direktori dari GDrive
!python train.py \
    --epochs 50 \
    --batch_size 4 \
    --lr 0.0001 \
    --noisy_dir "data/train/noisy" \
    --clean_dir "data/train/clean" \
    --val_noisy_dir "data/valid/noisy" \
    --val_clean_dir "data/valid/clean" \
    --checkpoint_dir "/content/drive/MyDrive/checkpoints/vitvs_denoising"



### 4. Evaluasi / Testing (Denoising)


In [ ]:
# Pilih checkpoint terbaik (misalnya yang memiliki val_loss terendah) atau checkpoint terakhir
ckpt_path = "/content/drive/MyDrive/checkpoints/vitvs_denoising/last.ckpt"

# Jalankan skrip test menggunakan dataset Test
!python test.py \
    --ckpt "$ckpt_path" \
    --input "data/test/noisy/sample_burung.wav" \
    --output "hasil_denoising.wav"

# Mainkan audio hasil denoising langsung di Colab
import IPython.display as ipd
ipd.Audio('hasil_denoising.wav')
